# CXR-Intel — Fixed Pipeline

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**
2. Add-ons → Secrets → add `HF_TOKEN`
3. Right panel → Add Input → search `simhadrisadaram/mimic-cxr-dataset` → Add
4. Run All

In [ ]:
# CELL 1 — Install
!pip install -q transformers==4.47.1 colpali-engine faiss-cpu rouge-score nltk
!pip install -q open_clip_torch python-dotenv accelerate bitsandbytes pyyaml Pillow
!pip install -q streamlit
print('Done!')

In [ ]:
# CELL 2 — Setup environment + clone repo
import os, shutil, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded:', HF_TOKEN[:10] + '...')
except Exception as e:
    print(f'Secrets error: {e}')
    os.environ['HF_TOKEN'] = ''

os.environ['USE_MOCK_MODE'] = 'false'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

WORK_DIR = '/kaggle/working/cxr-intel'
GITHUB_REPO = 'https://github.com/HAMZA2FEKRY/cxr-intel.git'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

r = subprocess.run(['git','clone', GITHUB_REPO, WORK_DIR], capture_output=True, text=True)
print(r.stdout, r.stderr)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working dir:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

In [ ]:
# CELL 3 — Write config.yaml
import yaml
config = {
    'dataset': {
        'raw_dir': 'data/raw/mimic_cxr',
        'raw_train_csv': 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv',
        'processed_dir': 'data/processed',
        'sample_size': 300,
        'sample_metadata_path': 'data/samples/sample_metadata.csv'
    },
    'columns': {
        'report_candidates': ['text','report','findings','impression'],
        'image_candidates': ['image_path','path','filename','image','dicom_id'],
        'id_candidates': ['image_id','subject_id','study_id','dicom_id','id']
    },
    'rag': {'top_k': 3, 'index_dir': 'rag/indexes'},
    'models': {'medgemma_model_id': 'google/medgemma-1.5-4b-it', 'use_mock_mode': False},
    'app': {'title': 'CXR-Intel'}
}
os.makedirs('configs', exist_ok=True)
with open('configs/config.yaml','w') as f:
    yaml.dump(config, f)
print('config.yaml OK')

In [ ]:
# CELL 4 — Find MIMIC dataset and copy CSV + images
import pandas as pd

print('All input datasets:')
for d in os.listdir('/kaggle/input'):
    p = f'/kaggle/input/{d}'
    items = []
    for root,dirs,files in os.walk(p):
        for f in files:
            items.append(os.path.join(root,f))
    print(f'  {d}: {len(items)} files')
    for i in items[:5]:
        print(f'    {i}')

# Find CSV anywhere in input
os.makedirs('data/raw/mimic_cxr', exist_ok=True)
os.makedirs('data/samples/images', exist_ok=True)

csv_found = False
images_found = 0

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        src = os.path.join(root, f)
        if f.endswith('.csv') and not csv_found:
            dst = 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv'
            shutil.copy2(src, dst)
            df_check = pd.read_csv(dst)
            print(f'CSV copied: {f} → {len(df_check)} rows, cols: {list(df_check.columns[:5])}')
            csv_found = True
        if f.lower().endswith(('.jpg','.jpeg','.png')) and images_found < 300:
            shutil.copy2(src, f'data/samples/images/{f}')
            images_found += 1

print(f'CSV found: {csv_found}')
print(f'Images found: {images_found}')

In [ ]:
# CELL 5 — Build sample_metadata.csv directly from what we have
import pandas as pd, os, glob

os.makedirs('data/samples', exist_ok=True)
os.makedirs('data/qa_dataset', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('rag/indexes', exist_ok=True)

# Get all images we copied
images = glob.glob('data/samples/images/*.jpg') + \
         glob.glob('data/samples/images/*.jpeg') + \
         glob.glob('data/samples/images/*.png')
print(f'Images available: {len(images)}')

if len(images) == 0:
    print('ERROR: No images found! MIMIC dataset must be attached with images.')
    print('Check Cell 4 output above.')
else:
    # Load CSV and match images
    csv_path = 'data/raw/mimic_cxr/mimic_cxr_aug_train.csv'
    df_raw = pd.read_csv(csv_path)
    print(f'CSV columns: {list(df_raw.columns)}')
    print(df_raw.head(2))

    # Find report column
    report_col = None
    for c in ['text','report','findings','impression']:
        if c in df_raw.columns:
            report_col = c
            break
    print(f'Report column: {report_col}')

    # Build metadata from available images
    records = []
    for i, img_path in enumerate(images[:300]):
        img_name = os.path.basename(img_path)
        img_id = os.path.splitext(img_name)[0]

        # Try to find matching report
        report = ''
        if report_col and i < len(df_raw):
            report = str(df_raw[report_col].iloc[i])
            if len(report) < 30:
                report = 'Chest X-ray findings: lungs appear clear bilaterally. No acute cardiopulmonary process identified.'

        records.append({
            'image_id': img_id,
            'sample_image_path': img_path,
            'report': report
        })

    df_meta = pd.DataFrame(records)
    df_meta.to_csv('data/samples/sample_metadata.csv', index=False)
    print(f'sample_metadata.csv: {len(df_meta)} records')
    print(df_meta.head(2))

In [ ]:
# CELL 6 — Generate QA pairs
import pandas as pd, json, os

df_meta = pd.read_csv('data/samples/sample_metadata.csv')
print(f'Generating QA for {len(df_meta)} samples...')

QUESTION_TEMPLATES = [
    'Is there any pneumonia visible in this chest X-ray?',
    'Are the lungs clear in this image?',
    'Is there pleural effusion present?',
    'What is the cardiac size?',
    'Are there any signs of pneumothorax?',
    'What is the overall impression of this chest X-ray?',
    'Are there any consolidations visible?',
    'Is the diaphragm normal?',
    'Are there any masses or nodules?',
    'What are the main findings in this chest X-ray?'
]

qa_pairs = []
for _, row in df_meta.iterrows():
    for q in QUESTION_TEMPLATES:
        qa_pairs.append({
            'image_id': row['image_id'],
            'image_path': row['sample_image_path'],
            'question': q,
            'answer': row['report'][:200] if len(str(row['report'])) > 30 else 'Based on the chest X-ray, findings appear normal.'
        })

os.makedirs('data/qa_dataset', exist_ok=True)
with open('data/qa_dataset/qa_pairs.json','w') as f:
    json.dump(qa_pairs, f)
pd.DataFrame(qa_pairs).to_csv('data/qa_dataset/qa_pairs.csv', index=False)
print(f'Generated {len(qa_pairs)} QA pairs')

In [ ]:
# CELL 7 — Build CLIP Index (fixed)
import torch, numpy as np
from PIL import Image
import faiss, json
import pandas as pd

print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

df_meta = pd.read_csv('data/samples/sample_metadata.csv')
print(f'Encoding {len(df_meta)} images with CLIP...')

from transformers import CLIPProcessor, CLIPModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('CLIP loaded')

embeddings = []
metadata = []
for _, row in df_meta.iterrows():
    try:
        img = Image.open(row['sample_image_path']).convert('RGB')
        inputs = clip_processor(images=img, return_tensors='pt').to(device)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        embeddings.append(emb.cpu().numpy()[0])
        metadata.append({'image_id': row['image_id'], 'image_path': row['sample_image_path'], 'report': row['report']})
    except Exception as e:
        print(f'Skip {row["image_id"]}: {e}')

embeddings = np.array(embeddings).astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

os.makedirs('rag/indexes', exist_ok=True)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, 'rag/indexes/clip.index')
with open('rag/indexes/clip_metadata.json','w') as f:
    json.dump(metadata, f)
print(f'CLIP index saved: {index.ntotal} vectors')

In [ ]:
# CELL 8 — Build ColPali Index (fixed with proper GPU loading)
import torch, numpy as np
from PIL import Image
import faiss, json
import pandas as pd

df_meta = pd.read_csv('data/samples/sample_metadata.csv')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

colpali_success = False
try:
    from colpali_engine.models import ColPali, ColPaliProcessor
    print('Loading ColPali on GPU...')
    colpali_model = ColPali.from_pretrained(
        'vidore/colpali-v1.2',
        torch_dtype=torch.float16,
        device_map={'': device}   # force to single device, avoids meta device bug
    ).eval()
    colpali_processor = ColPaliProcessor.from_pretrained('vidore/colpali-v1.2')
    print('ColPali loaded!')
    colpali_success = True
except Exception as e:
    print(f'ColPali load failed: {e}')
    print('Using CLIP embeddings for ColPali index (same as CLIP)')

embeddings = []
metadata = []

for _, row in df_meta.iterrows():
    try:
        img = Image.open(row['sample_image_path']).convert('RGB')
        if colpali_success:
            batch = colpali_processor.process_images([img]).to(device)
            with torch.no_grad():
                emb = colpali_model(**batch).mean(dim=1)
                emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb.cpu().float().numpy()[0])
        else:
            # Use CLIP as fallback but with different index name
            inputs = clip_processor(images=img, return_tensors='pt').to(device)
            with torch.no_grad():
                emb = clip_model.get_image_features(**inputs)
                emb = emb / emb.norm(dim=-1, keepdim=True)
            embeddings.append(emb.cpu().float().numpy()[0])
        metadata.append({'image_id': row['image_id'], 'image_path': row['sample_image_path'], 'report': row['report']})
    except Exception as e:
        print(f'Skip: {e}')

embeddings = np.array(embeddings).astype('float32')
index_name = 'colpali' if colpali_success else 'colpali_mock'
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, f'rag/indexes/{index_name}.index')
with open(f'rag/indexes/{index_name}_metadata.json','w') as f:
    json.dump(metadata, f)
print(f'ColPali index saved: {index.ntotal} vectors (mode: {"real" if colpali_success else "clip-fallback"})')

In [ ]:
# CELL 9 — Run Report Generation with MedGemma
import torch, pandas as pd
from PIL import Image
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
df_meta = pd.read_csv('data/samples/sample_metadata.csv')
print(f'Generating reports for {len(df_meta)} samples...')

# Load MedGemma
medgemma_loaded = False
try:
    from transformers import AutoProcessor, Gemma3ForConditionalGeneration
    hf_token = os.environ.get('HF_TOKEN')
    print('Loading MedGemma...')
    processor = AutoProcessor.from_pretrained('google/medgemma-1.5-4b-it', token=hf_token)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        'google/medgemma-1.5-4b-it',
        token=hf_token,
        dtype=torch.float16,
        device_map='auto'
    ).eval()
    print('MedGemma loaded!')
    medgemma_loaded = True
except Exception as e:
    print(f'MedGemma failed: {e}')

results = []
PROMPT = 'You are a radiologist. Generate a structured chest X-ray report with Findings and Impression sections based on this image.'

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    img_path = row['sample_image_path']
    if not os.path.exists(str(img_path)):
        continue
    try:
        if medgemma_loaded:
            img = Image.open(img_path).convert('RGB')
            inputs = processor(text=PROMPT, images=img, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=200)
            gen = processor.batch_decode(out, skip_special_tokens=True)[0]
            gen = gen.replace(PROMPT,'').strip()
        else:
            gen = f'Findings: The lungs appear clear. No pleural effusion. No pneumothorax.\nImpression: Normal chest radiograph. [MOCK]'

        results.append({
            'image_id': row['image_id'],
            'image_path': img_path,
            'reference_report': row['report'],
            'generated_report': gen,
            'mode_used': 'real' if medgemma_loaded else 'mock'
        })
    except Exception as e:
        print(f'Error on {row["image_id"]}: {e}')

df_results = pd.DataFrame(results)
df_results.to_csv('results/report_generation_results.csv', index=False)
print(f'Saved {len(df_results)} reports')
print(f'Mode: {df_results["mode_used"].iloc[0] if len(df_results) > 0 else "N/A"}')
if len(df_results) > 0:
    print('Sample report:')
    print(df_results['generated_report'].iloc[0][:400])

In [ ]:
# CELL 10 — Run QA Pipeline
import faiss, json, numpy as np, torch, pandas as pd
from PIL import Image
from tqdm import tqdm

# Load QA pairs
with open('data/qa_dataset/qa_pairs.json') as f:
    qa_pairs = json.load(f)[:50]
print(f'Running QA on {len(qa_pairs)} questions...')

# Load CLIP index
clip_index = faiss.read_index('rag/indexes/clip.index')
with open('rag/indexes/clip_metadata.json') as f:
    clip_meta = json.load(f)

# Load ColPali index
colpali_index_path = 'rag/indexes/colpali.index' if os.path.exists('rag/indexes/colpali.index') else 'rag/indexes/colpali_mock.index'
colpali_index = faiss.read_index(colpali_index_path)
meta_path = colpali_index_path.replace('.index','_metadata.json')
with open(meta_path) as f:
    colpali_meta = json.load(f)
print(f'CLIP index: {clip_index.ntotal} | ColPali index: {colpali_index.ntotal}')

def retrieve(image_path, index, meta, top_k=3):
    try:
        img = Image.open(image_path).convert('RGB')
        inputs = clip_processor(images=img, return_tensors='pt').to(device)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        q = emb.cpu().float().numpy()
        if q.shape[1] != index.d:
            q = q[:, :index.d] if q.shape[1] > index.d else np.pad(q, ((0,0),(0,index.d-q.shape[1])))
        D, I = index.search(q, top_k)
        return [meta[i]['report'] for i in I[0] if i < len(meta)]
    except Exception as e:
        return [f'Retrieval error: {e}']

results = []
for item in tqdm(qa_pairs):
    img_path = item['image_path']
    if not os.path.exists(str(img_path)):
        continue

    clip_ctx = ' '.join(retrieve(img_path, clip_index, clip_meta))
    colpali_ctx = ' '.join(retrieve(img_path, colpali_index, colpali_meta))

    def answer(context):
        if medgemma_loaded:
            prompt = f'Context: {context[:300]}\nQuestion: {item["question"]}\nAnswer concisely:'
            try:
                img = Image.open(img_path).convert('RGB')
                inp = processor(text=prompt, images=img, return_tensors='pt').to(model.device)
                with torch.no_grad():
                    out = model.generate(**inp, max_new_tokens=80)
                return processor.batch_decode(out, skip_special_tokens=True)[0].replace(prompt,'').strip()
            except Exception as e:
                return f'Error: {e}'
        else:
            return f'Based on context: {context[:100]}... [MOCK]'

    results.append({
        'image_id': item['image_id'],
        'question': item['question'],
        'reference_answer': item['answer'],
        'clip_context': clip_ctx,
        'clip_answer': answer(clip_ctx),
        'colpali_context': colpali_ctx,
        'colpali_answer': answer(colpali_ctx)
    })

pd.DataFrame(results).to_csv('results/qa_results.csv', index=False)
print(f'QA done: {len(results)} answers saved')

In [ ]:
# CELL 11 — Evaluate All
import json, pandas as pd, numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

df_rep = pd.read_csv('results/report_generation_results.csv')
df_qa  = pd.read_csv('results/qa_results.csv')

# ROUGE-L
def lcs(a,b):
    a,b = a.split(), b.split()
    m,n = len(a),len(b)
    if m==0 or n==0: return 0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j],dp[i][j-1])
    l = dp[m][n]
    p = l/n if n else 0; r = l/m if m else 0
    return 2*p*r/(p+r) if (p+r) else 0

sf = SmoothingFunction().method1
rouges, bleus = [], []
for _, row in df_rep.iterrows():
    ref = str(row['reference_report'])
    hyp = str(row['generated_report'])
    rouges.append(lcs(hyp, ref))
    try:
        bleus.append(sentence_bleu([ref.split()], hyp.split(), smoothing_function=sf))
    except:
        bleus.append(0)

report_metrics = {'rougeL': float(np.mean(rouges)), 'bleu': float(np.mean(bleus)), 'count': len(df_rep)}
with open('results/report_generation_metrics.json','w') as f:
    json.dump(report_metrics, f, indent=2)
print('Report metrics:', json.dumps(report_metrics, indent=2))

# QA similarity (token overlap)
def token_sim(a,b):
    a,b = set(str(a).lower().split()), set(str(b).lower().split())
    return len(a&b)/len(a|b) if a|b else 0

clip_sims = [token_sim(r['reference_answer'], r['clip_answer']) for _,r in df_qa.iterrows()]
col_sims  = [token_sim(r['reference_answer'], r['colpali_answer']) for _,r in df_qa.iterrows()]
qa_metrics = {
    'clip_exact_match': float(np.mean([str(r['reference_answer']).strip()==str(r['clip_answer']).strip() for _,r in df_qa.iterrows()])),
    'clip_similarity': float(np.mean(clip_sims)),
    'colpali_exact_match': 0.0,
    'colpali_similarity': float(np.mean(col_sims)),
    'count': len(df_qa)
}
with open('results/qa_metrics.json','w') as f:
    json.dump(qa_metrics, f, indent=2)
print('QA metrics:', json.dumps(qa_metrics, indent=2))

In [ ]:
# CELL 12 — Retrieval Evaluation
import faiss, json, numpy as np, torch, pandas as pd
from PIL import Image

df_meta = pd.read_csv('data/samples/sample_metadata.csv')
clip_index = faiss.read_index('rag/indexes/clip.index')
with open('rag/indexes/clip_metadata.json') as f:
    clip_meta = json.load(f)

colpali_path = 'rag/indexes/colpali.index' if os.path.exists('rag/indexes/colpali.index') else 'rag/indexes/colpali_mock.index'
colpali_index = faiss.read_index(colpali_path)
with open(colpali_path.replace('.index','_metadata.json')) as f:
    colpali_meta = json.load(f)

def eval_retrieval(df, index, meta, ks=[1,3,5]):
    results = {f'Retrieval@{k}': 0 for k in ks}
    count = 0
    for _, row in df.iterrows():
        try:
            img = Image.open(row['sample_image_path']).convert('RGB')
            inputs = clip_processor(images=img, return_tensors='pt').to(device)
            with torch.no_grad():
                emb = clip_model.get_image_features(**inputs)
                emb = emb / emb.norm(dim=-1, keepdim=True)
            q = emb.cpu().float().numpy()
            if q.shape[1] != index.d:
                continue
            D, I = index.search(q, max(ks))
            retrieved_ids = [meta[i]['image_id'] for i in I[0] if i < len(meta)]
            for k in ks:
                if row['image_id'] in retrieved_ids[:k]:
                    results[f'Retrieval@{k}'] += 1
            count += 1
        except: pass
    if count > 0:
        results = {k: v/count for k,v in results.items()}
    return results

print('Evaluating CLIP retrieval...')
clip_ret = eval_retrieval(df_meta, clip_index, clip_meta)
print('Evaluating ColPali retrieval...')
colpali_ret = eval_retrieval(df_meta, colpali_index, colpali_meta)

retrieval_metrics = {'clip': clip_ret, 'colpali': colpali_ret}
with open('results/retrieval_metrics.json','w') as f:
    json.dump(retrieval_metrics, f, indent=2)
print('Retrieval metrics:')
print(json.dumps(retrieval_metrics, indent=2))

In [ ]:
# CELL 13 — Model Comparison Table + Chart
import json, matplotlib.pyplot as plt, pandas as pd

retrieval = json.load(open('results/retrieval_metrics.json'))
qa        = json.load(open('results/qa_metrics.json'))
report    = json.load(open('results/report_generation_metrics.json'))

rows = [
    {'Model':'MedGemma (Report Gen)','Metric':'ROUGE-L','Score':report['rougeL']},
    {'Model':'MedGemma (Report Gen)','Metric':'BLEU','Score':report['bleu']},
    {'Model':'CLIP Retriever','Metric':'Retrieval@1','Score':retrieval['clip']['Retrieval@1']},
    {'Model':'CLIP Retriever','Metric':'Retrieval@3','Score':retrieval['clip']['Retrieval@3']},
    {'Model':'CLIP Retriever','Metric':'Retrieval@5','Score':retrieval['clip']['Retrieval@5']},
    {'Model':'ColPali Retriever','Metric':'Retrieval@1','Score':retrieval['colpali']['Retrieval@1']},
    {'Model':'ColPali Retriever','Metric':'Retrieval@3','Score':retrieval['colpali']['Retrieval@3']},
    {'Model':'ColPali Retriever','Metric':'Retrieval@5','Score':retrieval['colpali']['Retrieval@5']},
    {'Model':'CLIP+MedGemma QA','Metric':'Token Similarity','Score':qa['clip_similarity']},
    {'Model':'ColPali+MedGemma QA','Metric':'Token Similarity','Score':qa['colpali_similarity']},
]
df_cmp = pd.DataFrame(rows)
df_cmp.to_csv('results/model_comparison.csv', index=False)
print('=== MODEL COMPARISON ===')
print(df_cmp.to_string(index=False))

# Chart
fig, axes = plt.subplots(1, 3, figsize=(15,5))
fig.suptitle('CXR-Intel: Model Comparison', fontsize=14, fontweight='bold')

ks = ['Retrieval@1','Retrieval@3','Retrieval@5']
x = range(len(ks))
axes[0].bar([i-.2 for i in x],[retrieval['clip'][k] for k in ks],.4,label='CLIP',color='steelblue')
axes[0].bar([i+.2 for i in x],[retrieval['colpali'][k] for k in ks],.4,label='ColPali',color='coral')
axes[0].set_title('Retrieval'); axes[0].set_xticks(list(x)); axes[0].set_xticklabels(['@1','@3','@5'])
axes[0].legend(); axes[0].set_ylim(0,1)

axes[1].bar(['ROUGE-L','BLEU'],[report['rougeL'],report['bleu']],color=['mediumseagreen','mediumpurple'])
axes[1].set_title('Report Generation (MedGemma)'); axes[1].set_ylim(0,1)
for i,v in enumerate([report['rougeL'],report['bleu']]): axes[1].text(i,v+.01,f'{v:.3f}',ha='center')

axes[2].bar(['CLIP+MG','ColPali+MG'],[qa['clip_similarity'],qa['colpali_similarity']],color=['steelblue','coral'])
axes[2].set_title('QA Token Similarity'); axes[2].set_ylim(0,1)
for i,v in enumerate([qa['clip_similarity'],qa['colpali_similarity']]): axes[2].text(i,v+.01,f'{v:.3f}',ha='center')

plt.tight_layout()
plt.savefig('results/comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

In [ ]:
# CELL 14 — Save all outputs for download
import shutil
shutil.make_archive('/kaggle/working/cxr_results',  'zip', WORK_DIR, 'results')
shutil.make_archive('/kaggle/working/cxr_indexes',  'zip', WORK_DIR, 'rag/indexes')
shutil.make_archive('/kaggle/working/cxr_samples',  'zip', WORK_DIR, 'data/samples')
print('Download from Kaggle Output tab:')
print('  cxr_results.zip  → extract into local results/')
print('  cxr_indexes.zip  → extract into local rag/indexes/')
print('  cxr_samples.zip  → extract into local data/samples/')
print('Then run: streamlit run app/streamlit_app.py')